# Weaviate
     Its a fully featured, open-source vector database designed from the ground up for AI applications.Key Features: It goes beyond basic vector search by offering built-in hybrid search (combining sparse keyword vectors like BM25 with dense semantic vectors), built-in embedding generation, and multi-tenancy.

In [16]:
import weaviate
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from typing import TypedDict,Annotated,List,Any
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage,AIMessage,BaseMessage,SystemMessage
from langgraph.graph import StateGraph, START,END
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from langgraph.prebuilt import ToolNode
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command,interrupt
from sentence_transformers import SentenceTransformer
import numpy as np
import uuid
from chromadb.config import Settings
import chromadb
import os
load_dotenv()


True

In [17]:
# Best practice: store your credentials in environment variables
WEAVIATE_URL = os.environ["WEAVIATE_URL"]
WEAVIATE_API_KEY = os.environ["WEAVIATE_API_KEY"]
HF_TOKEN=os.environ["HF_TOKEN"]

In [18]:

client = weaviate.Client(
    url=WEAVIATE_URL, auth_client_secret=weaviate.AuthApiKey(WEAVIATE_API_KEY),
    additional_headers={
         "X-HuggingFace-Api-Key": HF_TOKEN
    },
)

# Your Weaviate class RAG schema shows it stores both:

- Raw text — the content property (your PDF page text) is kept verbatim, and Weaviate builds a BM25 inverted index on it (invertedIndexConfig.bm25).
Dense vector — Weaviate's text2vec-huggingface module automatically calls the HF inference API with sentence-transformers/all-MiniLM-L6-v2 to embed that same text, and stores the resulting vector in its HNSW-style index (vectorIndexType: hfresh, distance: cosine).
- you never called an embedding model yourself; the WeaviateHybridSearchRetriever + schema config did it for you implicitly.

In [19]:
schema = {
    "classes": [
        {
            "class": "RAG",
            "description": "Documents for RAG",
            "vectorizer": "text2vec-huggingface",
            "moduleConfig": {"text2vec-huggingface": {"model": "sentence-transformers/all-MiniLM-L6-v2", "type": "text"}},
            "properties": [
                {
                    "dataType": ["text"],
                    "description": "The content of the paragraph",
                    "moduleConfig": {
                        "text2vec-huggingface": {
                            "skip": False,
                            "vectorizePropertyName": False,
                        }
                    },
                    "name": "content",
                },
            ],
        },
    ]
}

  

client.schema.create(schema)
     

client.schema.get()

{'classes': [{'class': 'RAG',
   'description': 'Documents for RAG',
   'invertedIndexConfig': {'bm25': {'b': 0.75, 'k1': 1.2},
    'cleanupIntervalSeconds': 60,
    'stopwords': {'additions': None, 'preset': 'en', 'removals': None},
    'usingBlockMaxWAND': True},
   'moduleConfig': {'text2vec-huggingface': {'model': 'sentence-transformers/all-MiniLM-L6-v2',
     'type': 'text',
     'useCache': True,
     'useGPU': False,
     'vectorizeClassName': True,
     'waitForModel': False}},
   'multiTenancyConfig': {'autoTenantActivation': False,
    'autoTenantCreation': False,
    'enabled': False},
   'properties': [{'dataType': ['text'],
     'description': 'The content of the paragraph',
     'indexFilterable': True,
     'indexRangeFilters': False,
     'indexSearchable': True,
     'moduleConfig': {'text2vec-huggingface': {'skip': False,
       'vectorizePropertyName': False}},
     'name': 'content',
     'tokenization': 'word'}],
   'shardingConfig': {'actualCount': 1,
    'actualV

In [20]:
from langchain_community.retrievers import WeaviateHybridSearchRetriever

In [21]:

retriever = WeaviateHybridSearchRetriever(
    alpha = 0.5,               # defaults to 0.5, which is equal weighting between keyword and semantic search
    client = client,           # keyword arguments to pass to the Weaviate client
    index_name = "RAG",  # The name of the index to use
    text_key = "content",         # The name of the text key to use
    attributes=["source", "page", "total_pages", "title"], # The attributes to return in the results
    create_schema_if_missing=True,
)
     

In [22]:
from langchain_community.document_loaders import PyMuPDFLoader,DirectoryLoader

In [23]:
dirloader_pdf=DirectoryLoader('../data/pdf/',glob='**/*.pdf',loader_cls=PyMuPDFLoader)
docs_pdf=dirloader_pdf.load()
print(docs_pdf)

[Document(metadata={'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2025-07-25T11:02:56+05:30', 'source': '../data/pdf/IT314-Software Engineering-SPM Cont.pdf', 'file_path': '../data/pdf/IT314-Software Engineering-SPM Cont.pdf', 'total_pages': 10, 'format': 'PDF 1.5', 'title': 'Microsoft PowerPoint - IT314-Software Engineering-SPM Cont.ppt [Compatibility Mode]', 'author': 'DA-IICT', 'subject': '', 'keywords': '', 'moddate': '2025-07-25T11:02:56+05:30', 'trapped': '', 'modDate': "D:20250725110256+05'30'", 'creationDate': "D:20250725110256+05'30'", 'page': 0}, page_content='7/25/2025\n1\nDA-IICT\nIT 314: Software Engineering\nSoftware Process Models – RUP|XP|TDD\n1\nRUP – Rational Unified Process\n•\nLife Cycle model proposed by Booch, Jacobson, and Rumbaugh\n(“The three Amigos”) derived from the work on UML\n•\nRational Unified Process (RUP) uses Unified Modeling Language\n(UML) as core notation\n•\nDescribed from 3 perspective

In [24]:
retriever.add_documents(docs_pdf)

['d5b1e2b5-4a2a-4a64-8be5-04a23c431777',
 'eb39dd60-ecde-40c4-a5ac-e5df673c4e3d',
 '6ada16e4-74ca-4218-8fa5-7ad7f07cbbf6',
 'cab66c43-5c51-42b1-b64d-70c4c35fea68',
 '0a26f02f-af7b-4c62-908d-999849485c39',
 'f847ab3f-ece3-4baa-8477-9df0f6917f78',
 '4c5b5447-85c3-4460-a59e-cae0630e854d',
 'd22f202b-1a83-47ec-9024-6cde46f40993',
 'f46779a4-ad5b-44a0-80d5-f51c81756918',
 '7508bce1-7b9f-4b0b-960e-208d7e3cbcce']

In [25]:
res=retriever.invoke("Unified Process")
res

[Document(metadata={'page': 0, 'source': '../data/pdf/IT314-Software Engineering-SPM Cont.pdf', 'title': 'Microsoft PowerPoint - IT314-Software Engineering-SPM Cont.ppt [Compatibility Mode]', 'total_pages': 10}, page_content='7/25/2025\n1\nDA-IICT\nIT 314: Software Engineering\nSoftware Process Models – RUP|XP|TDD\n1\nRUP – Rational Unified Process\n•\nLife Cycle model proposed by Booch, Jacobson, and Rumbaugh\n(“The three Amigos”) derived from the work on UML\n•\nRational Unified Process (RUP) uses Unified Modeling Language\n(UML) as core notation\n•\nDescribed from 3 perspectives\n\uf0a7\n A dynamic perspective that shows phases over time;\n\uf0a7\n A static perspective that shows process activities;\n\uf0a7\n A practice perspective that suggests good practice.\n•\nUnified Process is distinguished by being\n\uf0a7\n Use-case driven\n\uf0a7\n Architecture-centric\n\uf0a7\n Iterative and incremental'),
 Document(metadata={'page': 4, 'source': '../data/pdf/IT314-Software Engineering-SPM